# CS1090B — Hallucination in Legal RAG Chatbots — End-to-End Pipeline

Runs the data pipeline described in the README on Google Colab Pro A100.
Stages: bootstrap → env check → CourtListener ingest → data audit → dataset probe → LePaRD ingest → LePaRD↔CL compat audit.

In [1]:
# Cell 1: Bootstrap repo + uv venv (pipeline cells will subprocess via .venv/bin/python)
"""
Purpose
-------
One-time environment bootstrap for the submission notebook. Clones the
project repo, installs the uv package manager, pins Python 3.11.9,
syncs the locked dependency tree into .venv, writes the reproducibility
.env file, and verifies that core ML libraries import at their pinned
versions.

What it sets up
---------------
- /content/cs1090b_HallucinationLegalRAGChatbots — repo clone (idempotent)
- uv installed at /root/.local/bin/uv (if not already present)
- Python 3.11.9 via `uv python install` + `uv python pin`
- .venv/ populated from uv.lock via `uv sync` (idempotent — skipped if
  .venv already exists)
- .env with reproducibility flags: PYTHONHASHSEED, CUBLAS_WORKSPACE_CONFIG,
  TOKENIZERS_PARALLELISM, RANDOM_SEED, TARGET_GPU_COUNT, TARGET_VRAM_GB_MIN

Verification
------------
After sync, imports numpy, torch, transformers from .venv and prints
their versions. Raises SystemExit if any import fails — fail-fast so
downstream cells don't run against a half-built environment.

Why subprocess-via-.venv
------------------------
Later pipeline cells invoke scripts as subprocesses using
.venv/bin/python rather than the Colab kernel's Python. This isolates
pinned versions from Colab's preinstalled (and frequently mismatched)
torch / transformers / numpy builds.

Runtime
-------
Cold start (fresh Colab runtime): ~3-6 min (git clone + uv install +
uv sync of full lockfile).
Warm start (repo already cloned, .venv intact): ~5-10s (verification
imports only).
"""

import os, subprocess, pathlib, time

def _fmt_elapsed(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    m, s = divmod(seconds, 60)
    if m < 60:
        return f"{int(m)}m {s:.1f}s"
    h, m = divmod(m, 60)
    return f"{int(h)}h {int(m)}m {s:.1f}s"

_t_start = time.perf_counter()

REPO_URL = "https://github.com/ltphongssvn/cs1090b_HallucinationLegalRAGChatbots.git"
REPO_DIR = "/content/cs1090b_HallucinationLegalRAGChatbots"

def sh(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, text=True)
    if check and r.returncode != 0:
        raise SystemExit(f"command failed: {cmd}")

if not pathlib.Path(REPO_DIR).exists():
    sh(f"git clone {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

if subprocess.run("command -v uv", shell=True).returncode != 0:
    sh("curl -LsSf https://astral.sh/uv/install.sh | sh")
    os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"

sh("uv python install 3.11.9")
sh("uv python pin 3.11.9")

if not pathlib.Path(".venv").exists():
    sh("uv sync")

pathlib.Path(".env").write_text(
    "export PYTHONHASHSEED=0\n"
    "export CUBLAS_WORKSPACE_CONFIG=:4096:8\n"
    "export TOKENIZERS_PARALLELISM=false\n"
    "export RANDOM_SEED=0\n"
    "export TARGET_GPU_COUNT=1\n"
    "export TARGET_VRAM_GB_MIN=20.0\n"
)

# Verify .venv has correct pinned versions
r = subprocess.run(
    [".venv/bin/python", "-c",
     "import sys, numpy, torch, transformers; "
     "print(f'python {sys.version.split()[0]}'); "
     "print(f'numpy {numpy.__version__}'); "
     "print(f'torch {torch.__version__}'); "
     "print(f'transformers {transformers.__version__}')"],
    capture_output=True, text=True,
)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr); raise SystemExit("venv verification failed")

_elapsed = time.perf_counter() - _t_start
print(f"⏱ Cell 1 - Bootstrap repo + uv venv completed in {_fmt_elapsed(_elapsed)}")

$ git clone https://github.com/ltphongssvn/cs1090b_HallucinationLegalRAGChatbots.git /content/cs1090b_HallucinationLegalRAGChatbots
cwd: /content/cs1090b_HallucinationLegalRAGChatbots
$ uv python install 3.11.9
$ uv python pin 3.11.9
$ uv sync
python 3.11.9
numpy 1.26.4
torch 2.0.1+cu117
transformers 4.41.2

⏱ Cell 1 - Bootstrap repo + uv venv completed in 1m 13.5s


# Login with a web browser, at Terminal prompt
git config --global user.email "ltphongssvn@gmail.com" && git config --global user.name "ltphongssvn"


gh auth login --hostname github.com --git-protocol https --web

In [2]:
# Cell 2: Environment verification — runs notebooks/cs1090b_HallucinationLegalRAGChatbots.md Cell 2
"""
Purpose
-------
Executes the environment verification cell from the canonical project
notebook (notebooks/cs1090b_HallucinationLegalRAGChatbots.md) inside
the pinned .venv, rather than the Colab kernel. Keeps the submission
notebook in sync with the canonical notebook's Cell 2 without
duplicating its source — the verification logic lives in exactly one
place and is extracted here via regex.

What it does
------------
1. Parses the canonical notebook's Markdown-formatted cells.
2. Extracts the first ```{code-cell}``` block (Cell 2 — environment
   verification: src.repro.configure, src.environment preflight checks,
   GPU / CUDA / VRAM validation, dependency version gates).
3. Patches out the canonical notebook's `os.chdir(..)` line since the
   submission notebook is already at the repo root.
4. Runs the patched code as a subprocess against .venv/bin/python with
   line-buffered streaming stdout so progress is visible in real time.
5. Suppresses the canonical notebook's inner "Cell 1 — Environment
   Setup" timer line so only this cell's outer timer is shown.
6. Raises SystemExit on non-zero return code.

Why subprocess-via-.venv
------------------------
The canonical Cell 2 imports torch, numpy, transformers, spacy at
their pinned versions. Running it in the Colab kernel would pick up
Colab's preinstalled (mismatched) builds. Routing through
.venv/bin/python guarantees the verification reflects the environment
downstream pipeline cells will actually use.

Runtime
-------
~15-45s depending on cold vs warm import cache and GPU preflight
latency.
"""

import subprocess, os, sys, re, pathlib, time

def _fmt_elapsed(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    m, s = divmod(seconds, 60)
    if m < 60:
        return f"{int(m)}m {s:.1f}s"
    h, m = divmod(m, 60)
    return f"{int(h)}h {int(m)}m {s:.1f}s"

_t_start = time.perf_counter()

os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")
nb_src = pathlib.Path("notebooks/cs1090b_HallucinationLegalRAGChatbots.md").read_text()
blocks = re.findall(r"```\{code-cell\}[^\n]*\n(.*?)```", nb_src, flags=re.DOTALL)
assert blocks, "no code cells found in notebook md"
cell1_code = blocks[0].replace(
    "os.chdir(os.path.dirname(os.getcwd()))", "# chdir not needed — already at repo root"
)
env = {**os.environ, "PYTHONUNBUFFERED": "1"}
proc = subprocess.Popen(
    [".venv/bin/python", "-u", "-c", cell1_code],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, text=True, env=env,
)
# Suppress the canonical notebook's inner timer line (labeled "Cell 1 —
# Environment Setup & GPU Verification") to avoid confusing double-timer
# output. Only the outer Cell 2 timer below is shown.
_inner_timer_re = re.compile(r"⏱\s*Cell 1\s*[—-].*completed in")
for line in proc.stdout:
    if _inner_timer_re.search(line):
        continue
    sys.stdout.write(line); sys.stdout.flush()
proc.wait()
if proc.returncode != 0:
    raise SystemExit(f"Cell 2 failed with exit code {proc.returncode}")

_elapsed = time.perf_counter() - _t_start
print(f"⏱ Cell 2 - Environment verification completed in {_fmt_elapsed(_elapsed)}")

  [repro] Reproducibility configured:
    PYTHONHASHSEED=0
    CUBLAS_WORKSPACE_CONFIG=:4096:8
    TOKENIZERS_PARALLELISM=false
    deterministic_algorithms=True
    cudnn_benchmark=False
    cudnn_deterministic=True
    random_seed=0
    torch.cuda.manual_seed_all(0) → 1 GPU(s)
    TDD RED→GREEN: Environment Contract
  ✓ PASS: Every required dependency must be importable and meet version constraints
  ✓ PASS: CUDA GPU must be detected for training
  ✓ PASS: GPU must have at least 10GB VRAM for transformer fine-tuning
  ✓ PASS: PyTorch must be compiled with CUDA support
  ✓ PASS: Cross-dependency version constraints must be satisfied
  
    Preflight Checks — Failure Isolation Gate
  ✓ PASS: GPU count 1 (allocated)
  ✓ PASS: GPU[0] NVIDIA A100-SXM4-80GB | cap (8, 0) | 85.1GB
  ✓ PASS: torch CUDA runtime 11.7
  ✓ PASS: Disk 201.9GB free
  ✓ PASS: src/repro.py importable
  ✓ PASS: repro_cfg['PYTHONHASHSEED'] = '0'
  ✓ PASS: repro_cfg['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
  ✓ PASS: repro

In [3]:
# Cell 3: Mount Google Drive and symlink CL bulk data directory
"""
Purpose
-------
Mounts Google Drive and wires the repo's data/raw/cl_bulk directory
to a Drive-persistent location so the CourtListener bulk CSV archives
survive Colab runtime resets. Cell 5's bulk_download stage then writes
directly into Drive, avoiding re-download on every session.

What it does
------------
1. force_remount Drive at /content/drive (fresh OAuth each session).
2. Reports Drive free/total space so the user can confirm ~60+ GB is
   available before attempting the CL bulk download.
3. Creates /content/drive/MyDrive/cs1090b_cl_bulk (idempotent).
4. Replaces (or creates) data/raw/cl_bulk as a symlink to the Drive
   directory. If the path already exists as an empty directory it is
   removed and re-linked; if already a symlink it is left alone.
5. Lists any existing .csv.bz2 archives in the Drive directory with
   sizes — useful to confirm a warm-start run already has the bulk
   archives from a prior session.

Drive layout
------------
/content/drive/MyDrive/cs1090b_cl_bulk/
    courts-*.csv.bz2
    dockets-*.csv.bz2
    opinion-clusters-*.csv.bz2
    opinions-*.csv.bz2

Symlinked into the repo as data/raw/cl_bulk/ so src.bulk_download and
downstream pipeline code see a local path and need no Drive-awareness.

Runtime
-------
~5-20s (dominated by Drive OAuth + mount; symlink ops are instant).
"""

import time, shutil, pathlib
from google.colab import drive

def _fmt_elapsed(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    m, s = divmod(seconds, 60)
    if m < 60:
        return f"{int(m)}m {s:.1f}s"
    h, m = divmod(m, 60)
    return f"{int(h)}h {int(m)}m {s:.1f}s"

_t_start = time.perf_counter()

# Mount Google Drive
drive.mount("/content/drive", force_remount=True)

# Check free space on Drive
total, used, free = shutil.disk_usage("/content/drive/MyDrive")
print(f"Drive free: {free/1e9:.1f} GB / total: {total/1e9:.1f} GB")

# Create target dir
dst = pathlib.Path("/content/drive/MyDrive/cs1090b_cl_bulk")
dst.mkdir(parents=True, exist_ok=True)
print(dst, "ready")

# Replace local dir with symlink to Drive
src = pathlib.Path("/content/cs1090b_HallucinationLegalRAGChatbots/data/raw/cl_bulk")
src.parent.mkdir(parents=True, exist_ok=True)
if src.is_symlink():
    print(f"already symlinked -> {src.resolve()}")
elif src.exists():
    src.rmdir()
    src.symlink_to(dst, target_is_directory=True)
    print(f"created {src} -> {dst}")
else:
    src.symlink_to(dst, target_is_directory=True)
    print(f"created {src} -> {dst}")

for p in sorted(dst.glob("*.csv.bz2")):
    print(f"  {p.name}  {p.stat().st_size/1e9:.2f} GB")

_elapsed = time.perf_counter() - _t_start
print(f"⏱ Cell 3 - Mount Drive and symlink cl_bulk completed in {_fmt_elapsed(_elapsed)}")

Mounted at /content/drive
Drive free: 191.8 GB / total: 259.7 GB
/content/drive/MyDrive/cs1090b_cl_bulk ready
created /content/cs1090b_HallucinationLegalRAGChatbots/data/raw/cl_bulk -> /content/drive/MyDrive/cs1090b_cl_bulk
  courts-2025-12-31.csv.bz2  0.00 GB
  dockets-2025-12-31.csv.bz2  4.88 GB
  opinion-clusters-2025-12-31.csv.bz2  2.45 GB
  opinions-2025-12-31.csv.bz2  53.70 GB
⏱ Cell 3 - Mount Drive and symlink cl_bulk completed in 21.0s


In [4]:
# Cell 4: CourtListener bulk CSV download — Drive-persistent, skip if present
"""
Purpose
-------
Downloads the 4 CourtListener bulk CSV archives (courts, dockets,
opinion-clusters, opinions) into the Drive-persistent directory
symlinked from Cell 3. Idempotent: if all 4 archives are already
present, skips the download entirely. Otherwise uses the pinned
2025-12-31 snapshot (matching Cell 5's manifest) via config.pinned_files
to download only what's missing.

What it does
------------
1. Ensures data/raw/cl_bulk symlink → /content/drive/MyDrive/cs1090b_cl_bulk
   (safety net in case Cell 3 was skipped).
2. Subprocesses into .venv/bin/python to run the pinned-env pipeline code:
   - PipelineConfig(pinned_*=...) — canonical paths and pinned 2025-12-31 keys
   - Check which of {courts-, dockets-, opinion-clusters-, opinions-}
     archives already exist in bulk_dir
   - If all 4 present: log sizes and skip
   - Otherwise: use cfg.pinned_files (no S3 discovery), then
     download_bulk_csvs() for the missing set, log destination paths
3. Streams subprocess output line-buffered so progress is visible in
   real time. Raises SystemExit on non-zero return code.

Snapshot pinning (2025-12-31)
-----------------------------
Pinning to 2025-12-31 matches the pre-processed shards + manifest
already on Drive (from the Harvard ODD L4 run). Without pinning, S3
discovery finds a newer 2026-03-31 snapshot and triggers a ~2-hour
cold re-extraction on Colab. With pinning, Cell 5's fast-path sees
matching source_files in the manifest and skips immediately.

Why subprocess-via-.venv
------------------------
src.config / src.s3_discovery / src.bulk_download pull boto3, polars,
and other pinned deps. Running in the Colab kernel would hit version
mismatches. .venv/bin/python guarantees the pinned environment.

Runtime
-------
Warm start (all archives in Drive): ~5-15s (directory scan + size log).
Cold start (full download): ~15-40 min depending on S3 throughput and
the total archive size (~60 GB compressed).
"""

import os, sys, subprocess, pathlib, time

def _fmt_elapsed(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    m, s = divmod(seconds, 60)
    if m < 60:
        return f"{int(m)}m {s:.1f}s"
    h, m = divmod(m, 60)
    return f"{int(h)}h {int(m)}m {s:.1f}s"

_t_start = time.perf_counter()

os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")
DRIVE_BULK = pathlib.Path("/content/drive/MyDrive/cs1090b_cl_bulk")
DRIVE_BULK.mkdir(parents=True, exist_ok=True)
local_bulk = pathlib.Path("data/raw/cl_bulk")
local_bulk.parent.mkdir(parents=True, exist_ok=True)
if local_bulk.exists() and not local_bulk.is_symlink():
    if local_bulk.is_dir() and not any(local_bulk.iterdir()):
        local_bulk.rmdir()
    else:
        raise SystemExit(f"{local_bulk} exists and is not empty/symlink")
if not local_bulk.exists():
    local_bulk.symlink_to(DRIVE_BULK, target_is_directory=True)
print(f"bulk_dir -> {local_bulk.resolve()}")

code = r'''
import logging
from src.config import PipelineConfig
from src.bulk_download import download_bulk_csvs
logging.basicConfig(level=logging.INFO, format="  %(message)s")
log = logging.getLogger("bulk")
cfg = PipelineConfig(
    pinned_courts="bulk-data/courts-2025-12-31.csv.bz2",
    pinned_dockets="bulk-data/dockets-2025-12-31.csv.bz2",
    pinned_clusters="bulk-data/opinion-clusters-2025-12-31.csv.bz2",
    pinned_opinions="bulk-data/opinions-2025-12-31.csv.bz2",
)
cfg.bulk_dir.mkdir(parents=True, exist_ok=True)
log.info("Snapshot: pinned 2025-12-31 (matches manifest on Drive)")
need = {"courts-", "dockets-", "opinion-clusters-", "opinions-"}
have = {p.name for p in cfg.bulk_dir.glob("*.csv.bz2")}
matched = {lbl for lbl in need if any(n.startswith(lbl) for n in have)}
if matched == need:
    log.info("All 4 bulk CSVs already present in %s - skipping" % cfg.bulk_dir)
    for p in sorted(cfg.bulk_dir.glob("*.csv.bz2")):
        log.info("  %s  %.2f GB" % (p.name, p.stat().st_size / 1e9))
else:
    log.info("Missing: %s" % sorted(need - matched))
    latest = cfg.pinned_files
    for label, info in latest.items():
        log.info("  %-10s %s" % (label, info["key"]))
    paths = download_bulk_csvs(latest, config=cfg, logger=log)
    for label, p in paths.items():
        log.info("  %-10s -> %s" % (label, p))
'''
env = {**os.environ, "PYTHONUNBUFFERED": "1"}
proc = subprocess.Popen(
    [".venv/bin/python", "-u", "-c", code],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, text=True, env=env,
)
for line in proc.stdout:
    sys.stdout.write(line); sys.stdout.flush()
proc.wait()
if proc.returncode != 0:
    raise SystemExit(f"Cell 4 failed with exit code {proc.returncode}")

_elapsed = time.perf_counter() - _t_start
print(f"⏱ Cell 4 - CourtListener bulk CSV download completed in {_fmt_elapsed(_elapsed)}")

bulk_dir -> /content/drive/MyDrive/cs1090b_cl_bulk
  Snapshot: pinned 2025-12-31 (matches manifest on Drive)
  All 4 bulk CSVs already present in data/raw/cl_bulk - skipping
    courts-2025-12-31.csv.bz2  0.00 GB
    dockets-2025-12-31.csv.bz2  4.88 GB
    opinion-clusters-2025-12-31.csv.bz2  2.45 GB
    opinions-2025-12-31.csv.bz2  53.70 GB
⏱ Cell 4 - CourtListener bulk CSV download completed in 0.2s


In [6]:
# Cell 5: Filter chain + extraction + manifest (Drive-persistent shards)
"""
Purpose
-------
End-to-end Stage 1-2 data pipeline: pinned snapshot → filter chain →
extract → manifest → TDD contract tests. Produces the Drive-persistent
NDJSON shards that Cells 5.5, 6, and 9 operate on.

Fast-path behavior
------------------
If manifest.json and all referenced shards are already present in the
Drive-persistent shard directory (e.g. pre-processed on Harvard ODD
GPU Cluster L4 and copied to Drive), run_pipeline() returns immediately
without re-running any stage. Otherwise runs the full pipeline:
  1. Pinned snapshot — use cfg.pinned_files (no S3 discovery)
  2. Download        — fetch missing archives into data/raw/cl_bulk
  3. Filter chain    — court → docket → cluster → opinion joins,
                       restricted to federal appellate circuits
  4. Extract         — per-opinion text extraction (html/plain/xml),
                       quarantine for per-row failures
  5. Manifest        — write manifest.json with provenance
                       (git rev, snapshot id, court distribution, stats)
  6. Contract tests  — validate_pipeline() runs TDD invariants
                       against the manifest and a sample of shards

Snapshot pinning (2025-12-31)
-----------------------------
PipelineConfig is pinned to the 2025-12-31 CL bulk snapshot, which
matches the pre-processed shards + manifest already on Drive from the
Harvard ODD L4 run. Without pinning, run_pipeline calls
discover_latest_bulk_files() which finds the newer 2026-03-31 snapshot
on S3, sees the manifest's source_files don't match, and triggers a
~2-hour cold re-extraction on Colab's 12-core CPU. With pinning, the
fast-path check succeeds immediately and the cell returns in <30s.

Drive layout
------------
/content/drive/MyDrive/cs1090b_cl_federal_appellate_bulk/
    shard_0000.jsonl ... shard_NNNN.jsonl
    manifest.json
Symlinked into the repo as data/raw/cl_federal_appellate_bulk/ so
downstream cells see a local path.

Runtime
-------
Warm start (shards + manifest valid, pinned): ~10-30s (manifest parse +
contract tests on sampled shards).
Cold start (full pipeline from pinned snapshot): ~30-90 min on a
high-core machine; ~2+ hours on Colab's 12-core runtime.
"""

import os, pathlib
os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

# Symlink shard_dir to Drive so shards persist across Colab sessions
DRIVE_SHARDS = pathlib.Path("/content/drive/MyDrive/cs1090b_cl_federal_appellate_bulk")
DRIVE_SHARDS.mkdir(parents=True, exist_ok=True)
local_shards = pathlib.Path("data/raw/cl_federal_appellate_bulk")
local_shards.parent.mkdir(parents=True, exist_ok=True)
if local_shards.exists() and not local_shards.is_symlink():
    if local_shards.is_dir() and not any(local_shards.iterdir()):
        local_shards.rmdir()
    else:
        raise RuntimeError(f"{local_shards} exists and is not empty/symlink")
if not local_shards.exists():
    local_shards.symlink_to(DRIVE_SHARDS, target_is_directory=True)
print(f"shard_dir -> {local_shards.resolve()}")

# In-process pipeline — no subprocess, no embedded code string
import sys, logging
from src.config import PipelineConfig
from src.pipeline import run_pipeline, validate_pipeline
from src.exceptions import PipelineError
from src.timer import cell_timer

def _get_cell5_logger():
    lg = logging.getLogger("cell5")
    lg.setLevel(logging.INFO)
    if not lg.handlers:  # idempotent on cell re-run — fixes duplicate-handler bug
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(logging.Formatter("  %(message)s"))
        lg.addHandler(h)
    lg.propagate = False
    return lg

logger = _get_cell5_logger()
cfg = PipelineConfig(
    pinned_courts="bulk-data/courts-2025-12-31.csv.bz2",
    pinned_dockets="bulk-data/dockets-2025-12-31.csv.bz2",
    pinned_clusters="bulk-data/opinion-clusters-2025-12-31.csv.bz2",
    pinned_opinions="bulk-data/opinions-2025-12-31.csv.bz2",
)

with cell_timer("Cell 5 - Pipeline (filter + extract + manifest)", logger=logger):
    logger.info("=" * 60)
    logger.info("  run_pipeline (pinned 2025-12-31; fast-path if manifest + shards valid)")
    logger.info("=" * 60)
    manifest = run_pipeline(config=cfg, logger=logger)
    logger.info("\n" + "=" * 60)
    logger.info("  validate_pipeline (TDD contract tests)")
    logger.info("=" * 60)
    validate_pipeline(
        config=cfg, manifest_data=manifest,
        logger=logger, shard_strategy="sample",
    )
    logger.info("OK contract tests passed")
    logger.info("\n" + "=" * 60)
    logger.info("  Summary")
    logger.info("=" * 60)
    logger.info("  snapshot: %s" % manifest["source_files"]["opinions"])
    logger.info("  git_rev:  %s" % manifest["run_metadata"]["git_revision"][:12])
    logger.info("  shards:   %d" % manifest["num_shards"])
    logger.info("  cases:    %s" % format(manifest["num_cases"], ","))
    logger.info("  scanned:  %s" % format(manifest.get("scanned", 0), ","))
    logger.info("  circuits: %d" % len(manifest.get("court_distribution", {})))
    tls = manifest.get("text_length_stats", {})
    if tls:
        logger.info("  text len: mean=%s median=%s p95=%s" % (
            format(tls.get("mean", 0), ","),
            format(tls.get("median", 0), ","),
            format(tls.get("p95", 0), ","),
        ))

shard_dir -> /content/drive/MyDrive/cs1090b_cl_federal_appellate_bulk
    run_pipeline (pinned 2025-12-31; fast-path if manifest + shards valid)
  ✓ Already complete: 1,465,484 cases, 159 shards verified
  
    validate_pipeline (TDD contract tests)
  ✓ PASS: Shard directory must exist
  ✓ PASS: Manifest must exist
  ✓ PASS: At least one shard present
  ✓ PASS: Sufficient opinions
  ✓ PASS: Valid JSON
  ✓ PASS: Text present
  ✓ PASS: Text substantive
  ✓ PASS: Provenance metadata
  ✓ PASS: Raw + normalized text + flags
  ✓ PASS: Text source tracked
  ✓ PASS: Multiple circuits
  ✓ PASS: Schema consistent
  ✓ PASS: Checksums match
  OK contract tests passed
  
    Summary
    snapshot: opinions-2025-12-31.csv.bz2
    git_rev:  780ff292fbd6
    shards:   159
    cases:    1,465,484
    scanned:  10,682,555
    circuits: 13
    text len: mean=12,506 median=6,373 p95=43,952
  ⏱ Cell 5 - Pipeline (filter + extract + manifest) completed in 21m 34.6s


In [7]:
# Cell 5.5: Upstream NaN repair — makes Cell 6 Polars fast-path work on all shards
"""
Purpose
-------
Repairs bare NaN/Infinity tokens in Cell 5's NDJSON shards so Cell 6's
Polars scan_ndjson fast-path works on all shards. Python's json writer
emits bare NaN by default (non-spec JSON); Polars' strict simd-json
parser rejects these with TapeError, forcing a slower fallback that
silently drops records. This cell fixes the shards upstream of Cell 6
so all records are loaded via Polars' fast path.

Three-step flow
---------------
1. Audit before repair: scans all shards, identifies contaminated ones
   (typically 8 of 159 for the CL federal appellate corpus), reports
   pre-repair verdict and nan_lines count.
2. Repair: runs scripts/audit_jsonl_nan.py --fix --parallel-repair
   --validate. Semantic repair (json.loads with parse_constant intercept
   → recursive NaN→None walk → json.dumps(allow_nan=False)) so legal
   text containing literal "NaN" inside quoted strings is never
   corrupted. Streaming (O(1) RAM regardless of shard size), atomic
   rename with .bak backup, idempotent, Polars-validated post-repair.
3. Re-audit: confirms post-repair verdict is CLEAN and clean_pct=100.0.
   Raises RuntimeError on anything other than CLEAN.

Why this cell exists
--------------------
Without it, Cell 6 emits 8 Polars TapeError WARNINGs and loads only
~1,381,584 of ~1,465,484 records (silently dropping ~83,900). With it,
Cell 6 takes the fast path on all shards and loads the full corpus.

Runtime
-------
~8-12 min on 12-core Colab A100 runtime (audit ~90s × 2 + repair ~5min
over 159 shards averaging ~9 MB each). Re-runs after a first successful
repair are ~3 min (audit still scans but finds nothing to repair).
"""

import os, sys, subprocess, json, pathlib, re, time

def _fmt_elapsed(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    m, s = divmod(seconds, 60)
    if m < 60:
        return f"{int(m)}m {s:.1f}s"
    h, m = divmod(m, 60)
    return f"{int(h)}h {int(m)}m {s:.1f}s"

_t_start = time.perf_counter()

os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")
shard_dir = pathlib.Path("data/raw/cl_federal_appellate_bulk")
if not shard_dir.exists() or not any(shard_dir.glob("shard_*.jsonl")):
    raise RuntimeError(f"no shards under {shard_dir} — run Cell 5 first")

def _stream(cmd):
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            bufsize=1, text=True, env=env)
    lines = []
    for line in proc.stdout:
        sys.stdout.write(line); sys.stdout.flush()
        lines.append(line)
    proc.wait()
    return proc.returncode, "".join(lines)

def _extract_report(out: str) -> dict:
    # audit_jsonl_nan.py emits json.dumps(..., indent=2), so the top-level
    # object has '{' and '}' anchored at column 0. Match line-anchored braces
    # to avoid false matches on nested '{}' (e.g. empty "nan_fields": {}).
    match = re.search(r'^\{.*?^\}', out, re.MULTILINE | re.DOTALL)
    if not match:
        raise RuntimeError("could not locate JSON report in audit output")
    return json.loads(match.group(0))

print("=" * 60)
print("  Step 1: audit before repair")
print("=" * 60)
rc, out = _stream([".venv/bin/python", "scripts/audit_jsonl_nan.py",
                   "--input-dir", str(shard_dir), "--json", "--emit-shard-ids"])
if rc != 0:
    raise SystemExit(f"pre-repair audit failed with exit code {rc}")
pre_report = _extract_report(out)
print(f"\n  pre-repair verdict: {pre_report.get('gate_verdict')}")
print(f"  pre-repair nan_lines: {pre_report.get('nan_lines')}")

print("\n" + "=" * 60)
print("  Step 2: repair (semantic, parallel, Polars-validated)")
print("=" * 60)
rc, _ = _stream([".venv/bin/python", "scripts/audit_jsonl_nan.py",
                 "--input-dir", str(shard_dir),
                 "--fix", "--parallel-repair", "--validate"])
if rc != 0:
    raise SystemExit(f"repair failed with exit code {rc}")

print("\n" + "=" * 60)
print("  Step 3: re-audit after repair (must be CLEAN)")
print("=" * 60)
rc, out = _stream([".venv/bin/python", "scripts/audit_jsonl_nan.py",
                   "--input-dir", str(shard_dir), "--json"])
if rc != 0:
    raise SystemExit(f"post-repair audit failed with exit code {rc}")
report = _extract_report(out)
if report.get("gate_verdict") != "CLEAN":
    raise RuntimeError(f"post-repair verdict not CLEAN: {report.get('gate_verdict')}")
print(f"\nOK repair complete — verdict: {report['gate_verdict']}, "
      f"clean_pct: {report['clean_pct']}, total_lines: {report['total_lines']:,}")

_elapsed = time.perf_counter() - _t_start
print(f"⏱ Cell 5.5 - Upstream NaN repair completed in {_fmt_elapsed(_elapsed)}")

  Step 1: audit before repair
[INFO] Scanning 159 shards using 12 CPU cores ...

auditing: 100%|██████████| 159/159 [01:27<00:00,  1.82shard/s]
{
  "total_lines": 1456611,
  "nan_lines": 1992,
  "nonfinite_lines": 1992,
  "string_sentinel_lines": 0,
  "decode_error_lines": 0,
  "nan_shards": 9,
  "total_shards": 159,
  "clean_pct": 99.8632,
  "nan_fields": {
    "case_name": 1992
  },
  "gate_verdict": "REPAIRABLE \u2014 NaN only in advisory fields; does not block Stage 3",
  "contaminated_shards": [
    "shard_0000.jsonl",
    "shard_0001.jsonl",
    "shard_0002.jsonl",
    "shard_0003.jsonl",
    "shard_0004.jsonl",
    "shard_0005.jsonl",
    "shard_0006.jsonl",
    "shard_0007.jsonl",
    "shard_0009.jsonl"
  ]
}

  pre-repair verdict: REPAIRABLE — NaN only in advisory fields; does not block Stage 3
  pre-repair nan_lines: 1992

  Step 2: repair (semantic, parallel, Polars-validated)
[INFO] REPAIRING 159 shards (parallel=True, workers=12) ...

repairing: 100%|██████████| 159/159 [0

In [13]:
# Cell 6: Dataset readiness probe — full-corpus Polars scan, 8 gates
"""
Purpose
-------
Dataset readiness gate for the CourtListener federal appellate corpus
ingested in Cell 5 and cleaned in Cell 5.5. Runs src.dataset_probe
(v2.5.11+, 303 contract tests, frozen ProbeConfig) as a full-corpus
Polars scan with 8 quality gates. Go/no-go check before Stage 3.

Gates evaluated
---------------
  schema — Pydantic schema validation on every record
  A7     — text_source breakdown (html vs plain_text vs xml distribution)
  A8     — text_length distribution (blocking — enforces min/max bounds)
  A9     — citation_count distribution (advisory)
  A11    — tokenizer-aware chunk count under BAAI/bge-m3 (blocking)
  A12    — citation anchor survival after text extraction (blocking)
  A13    — sentence density via spaCy (blocking)
  B6     — text_entropy distribution (advisory — detects degenerate text)

Blocking gates must PASS for the corpus to clear Stage 3. Advisory gates
surface warnings but do not fail the run.

HF_TOKEN
--------
Gate A11 downloads the BAAI/bge-m3 tokenizer from Hugging Face. The
model is public so a token is not strictly required, but Hugging Face
Hub emits a UserWarning when HF_TOKEN is unset. This cell loads the
token from Colab secrets if available to silence the warning.

Summary counts
--------------
dataset_probe v2.5.12 does not populate summary["passed_count"] or
summary["failed_blocking_count"], so this cell derives them directly
from report.gates as ground truth. Avoids the cosmetic "passed_count: 0"
bug seen in prior runs.

Performance
-----------
Full scan uses Polars scan_ndjson (memory-mapped, lazy). First run on a
cold corpus: ~8-10 min for 1.46M rows on a 12-core Colab A100 runtime.
Re-runs are cheap (~30s) because Polars mmaps the shards. Requires
Cell 5.5 to have repaired any bare-NaN shards — otherwise contaminated
shards fall back to a slower Python path and some records are dropped.

Output
------
logs/dataset_probe_report.json — full ProbeReport dump (gates, summary,
provenance, per-gate evidence). Raises RuntimeError if any blocking
gate fails.

Runtime
-------
Full scan : ~9-10 min
"""
import os, sys, logging, json, pathlib

os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

# Load HF_TOKEN from Colab secrets to silence huggingface_hub UserWarning
# emitted by Gate A11's BAAI/bge-m3 tokenizer download. Public model, so
# a token is not required for access — this purely suppresses the warning.
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass  # not on Colab, or secret not set — warning will appear but probe still runs

from src.dataset_probe import run_probe, ProbeConfig
from src.timer import cell_timer


def _get_cell6_logger():
    lg = logging.getLogger("cell6")
    lg.setLevel(logging.INFO)
    if not lg.handlers:
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(logging.Formatter("  %(message)s"))
        lg.addHandler(h)
    lg.propagate = False
    return lg


logger = _get_cell6_logger()
shard_dir = pathlib.Path("data/raw/cl_federal_appellate_bulk")
out_path = pathlib.Path("logs/dataset_probe_report.json")
out_path.parent.mkdir(parents=True, exist_ok=True)

with cell_timer("Cell 6 - Dataset readiness probe", logger=logger):
    logger.info("=" * 60)
    logger.info("  run_probe (full-corpus Polars scan, 8 gates)")
    logger.info("=" * 60)
    logger.info("  shard_dir: %s" % shard_dir.resolve())

    probe_cfg = ProbeConfig()
    report = run_probe(
        data_dir=shard_dir,
        subset=1_465_484,     # full corpus
        output=out_path,      # run_probe writes report to this path
        full_scan=True,       # always True in v2.5.11+ (Polars mandatory)
        config=probe_cfg,
    )
    logger.info("  report -> %s" % out_path)

    logger.info("\n" + "=" * 60)
    logger.info("  Probe summary")
    logger.info("=" * 60)

    summary = (
        report.summary
        if isinstance(report.summary, dict)
        else report.summary.model_dump()
    )

    # Compute gate counts directly from report.gates — dataset_probe v2.5.12
    # does not populate summary["passed_count"] / summary["failed_blocking_count"],
    # so derive them from the per-gate results as ground truth.
    gate_items = [
        (name, gr if isinstance(gr, dict) else gr.model_dump())
        for name, gr in report.gates.items()
    ]
    total_gates = len(gate_items)
    passed_count = sum(1 for _, gr in gate_items if gr.get("pass"))
    failed_blocking = sum(
        1 for _, gr in gate_items
        if not gr.get("pass") and gr.get("severity") == "blocking"
    )
    failed_advisory = sum(
        1 for _, gr in gate_items
        if not gr.get("pass") and gr.get("severity") == "advisory"
    )

    logger.info("  all_passed:       %s" % summary.get("all_passed"))
    logger.info("  passed_count:     %d / %d" % (passed_count, total_gates))
    logger.info("  failed_blocking:  %d" % failed_blocking)
    logger.info("  failed_advisory:  %d" % failed_advisory)
    logger.info("  subset_n:         %s" % format(report.subset_n, ","))
    logger.info("  parse_errors:     %d" % getattr(report, "parse_errors", 0))

    logger.info("\n  Gate results:")
    for gate_name, gr in gate_items:
        status = "PASS" if gr.get("pass") else "FAIL"
        sev = gr.get("severity", "?")
        logger.info("    %-10s %s  (%s)" % (gate_name, status, sev))

    prov = getattr(report, "provenance", {}) or {}
    if not isinstance(prov, dict):
        prov = prov.model_dump()
    logger.info("\n  probe_version:  %s" % prov.get("probe_version"))
    logger.info("  polars_version: %s" % prov.get("polars_version"))
    logger.info("  full_scan:      %s" % prov.get("full_scan"))

    # Blocking-failure gate: derived from gates, not summary, to avoid
    # depending on summary["failed_blocking_count"] which is not populated.
    if failed_blocking > 0:
        raise RuntimeError(
            f"probe blocking gates failed ({failed_blocking}) — see {out_path}"
        )
    if not summary.get("all_passed"):
        logger.info(
            "  NOTE: summary.all_passed is False but no blocking gates failed "
            "(likely an advisory gate) — continuing."
        )

    logger.info("\nOK all blocking gates passed — corpus cleared for Stage 3")

    run_probe (full-corpus Polars scan, 8 gates)
    shard_dir: /content/drive/MyDrive/cs1090b_cl_federal_appellate_bulk
[dataset_probe] Full scan mode — loading all records from data/raw/cl_federal_appellate_bulk via Polars ...
[dataset_probe] Full scan loaded 1456611 records.
[dataset_probe] Gate: schema validation ...
[dataset_probe] Gate A7: text_source breakdown ...
[dataset_probe] Gate A8: text_length distribution ...
[dataset_probe] Gate A9: citation_count distribution ...
[dataset_probe] Gate A12: citation anchor survival ...
[dataset_probe] Gate B6: text_entropy distribution ...
[dataset_probe] Gate A11: tokenizer-aware chunk count (BAAI/bge-m3) ...


Token indices sequence length is longer than the specified maximum sequence length for this model (9214 > 8192). Running this sequence through the model will result in indexing errors


[dataset_probe] Gate A13: sentence density (spaCy) ...
[dataset_probe] Quality signals ...
[dataset_probe] Report written → logs/dataset_probe_report.json
[dataset_probe] PASSED: ['schema', 'A7', 'A8', 'A9', 'A12', 'B6', 'A11', 'A13'] | FAILED_BLOCKING: [] | FAILED_ADVISORY: [] | SKIPPED: []
    report -> logs/dataset_probe_report.json
  
    Probe summary
    all_passed:       True
    passed_count:     8 / 8
    failed_blocking:  0
    failed_advisory:  0
    subset_n:         1,456,611
    parse_errors:     0
  
  Gate results:
      schema     PASS  (blocking)
      A7         PASS  (blocking)
      A8         PASS  (blocking)
      A9         PASS  (advisory)
      A12        PASS  (blocking)
      B6         PASS  (advisory)
      A11        PASS  (blocking)
      A13        PASS  (blocking)
  
  probe_version:  2.5.12
    polars_version: 1.25.2
    full_scan:      True
  
OK all blocking gates passed — corpus cleared for Stage 3
  ⏱ Cell 6 - Dataset readiness probe completed in 

In [9]:
# Cell 7: LePaRD dataset ingestion (Drive-persistent, self-healing)
"""
Purpose
-------
Ensures the LePaRD gold-standard citation dataset is present and verified
in Drive-persistent storage, ready for Cell 8's compatibility audit.
Uses scripts/ingest_lepard.py — pinned HF revision, atomic write,
sidecar + manifest provenance.

Three-step flow
---------------
1. --verify-only fast-path: checks existing Drive artifact (Harvard ODD
   copy). If SHA256 + sidecar + manifest all valid → skip to done.
2. Self-heal / re-ingest: if verify fails, run default ingest which is
   idempotent — recomputes SHA256 from existing JSONL bytes, rebuilds
   missing sidecar (.sha256) and manifest.json with
   provenance_reconstructed=True. Only triggers an HF download if the
   JSONL itself is absent or its SHA256 doesn't match the pinned revision.
3. --verify-only post-ingest: confirms final state is valid.

Drive layout
------------
/content/drive/MyDrive/cs1090b_lepard/  (persistent across Colab sessions)
    lepard_train_4000000_rev0194f95.jsonl
    lepard_train_4000000_rev0194f95.jsonl.sha256
    manifest.json
Symlinked into the repo as data/raw/lepard/ for tool compatibility.

HF_TOKEN
--------
Loaded lazily from Colab secrets only if Step 2 triggers a re-download.
Step 1 (fast-path verify) never needs it.

Runtime
-------
Fast-path (Step 1 only): ~1-2 min (SHA256 over 5.8 GB + JSON parse).
Full re-ingest with download: ~10-20 min depending on HF bandwidth.
"""

import os, sys, subprocess, pathlib, time
os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

# Symlink data/raw/lepard to Drive
DRIVE_LEPARD = pathlib.Path("/content/drive/MyDrive/cs1090b_lepard")
DRIVE_LEPARD.mkdir(parents=True, exist_ok=True)
local_lepard = pathlib.Path("data/raw/lepard")
local_lepard.parent.mkdir(parents=True, exist_ok=True)
if local_lepard.exists() and not local_lepard.is_symlink():
    if local_lepard.is_dir() and not any(local_lepard.iterdir()):
        local_lepard.rmdir()
    else:
        raise RuntimeError(f"{local_lepard} exists and is not empty/symlink")
if not local_lepard.exists():
    local_lepard.symlink_to(DRIVE_LEPARD, target_is_directory=True)
print(f"lepard_dir -> {local_lepard.resolve()}")

# HF_TOKEN loaded lazily — only required if re-download triggers
def _load_hf_token():
    if os.environ.get("HF_TOKEN"):
        return True
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("HF_TOKEN loaded from Colab secrets")
        return True
    except Exception:
        return False

def _stream(cmd):
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, text=True, env=env,
    )
    for line in proc.stdout:
        sys.stdout.write(line); sys.stdout.flush()
    proc.wait()
    return proc.returncode

def _fmt_elapsed(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    m, s = divmod(seconds, 60)
    if m < 60:
        return f"{int(m)}m {s:.1f}s"
    h, m = divmod(m, 60)
    return f"{int(h)}h {int(m)}m {s:.1f}s"

_t_start = time.perf_counter()

INGEST = [".venv/bin/python", "scripts/ingest_lepard.py"]

# Step 1: fast-path verify (no HF_TOKEN needed)
print("\n" + "=" * 60)
print("  Step 1: --verify-only (fast-path)")
print("=" * 60)
rc = _stream(INGEST + ["--verify-only"])
if rc == 0:
    print("\nOK artifact valid — skipping ingest (Harvard ODD copy in good state)")
else:
    # Step 2: self-heal OR re-ingest
    # Default invocation: idempotent, recomputes SHA256 from disk bytes,
    # reconstructs missing sidecar/manifest with provenance_reconstructed=True,
    # only downloads from HF if JSONL is absent or SHA256 mismatch.
    print("\n" + "=" * 60)
    print("  Step 2: self-heal / re-ingest")
    print("=" * 60)
    print("  ingest_lepard.py will:")
    print("    - recompute SHA256 from existing JSONL bytes")
    print("    - rebuild missing sidecar (.sha256) from computed digest")
    print("    - rebuild missing manifest.json with provenance_reconstructed=True")
    print("    - only download from HF if JSONL itself is missing or digest mismatch")
    if not _load_hf_token():
        print("  NOTE: HF_TOKEN not set — download path will fail if triggered")
    rc = _stream(INGEST)
    if rc != 0:
        raise SystemExit(f"LePaRD ingest failed with exit code {rc}")

    # Step 3: post-ingest verify
    print("\n" + "=" * 60)
    print("  Step 3: --verify-only (post-ingest confirmation)")
    print("=" * 60)
    rc = _stream(INGEST + ["--verify-only"])
    if rc != 0:
        raise SystemExit("post-ingest verify failed")

print("\nOK LePaRD artifact ready")
for p in sorted(DRIVE_LEPARD.glob("*")):
    size = p.stat().st_size
    print(f"  {p.name}  {size/1e9:.2f} GB" if size > 1e6 else f"  {p.name}  ({size} B)")

_elapsed = time.perf_counter() - _t_start
print(f"\n⏱ Cell 7 - LePaRD ingestion completed in {_fmt_elapsed(_elapsed)}")

lepard_dir -> /content/drive/MyDrive/cs1090b_lepard

  Step 1: --verify-only (fast-path)
[INFO] LePaRD ingestion — dataset=rmahari/LePaRD revision=0194f95c3091acceab3b887c9b09ef432cf84052 cap=4000000 force=False dry_run=False verify_only=True
[INFO] Verified lepard_train_4000000_rev0194f95.jsonl — digest matches sidecar
[INFO] Verified lepard_train_4000000_rev0194f95.jsonl — manifest fields match
[INFO] sha256=abe787c0...
[INFO] Done — data/raw/lepard/lepard_train_4000000_rev0194f95.jsonl

OK artifact valid — skipping ingest (Harvard ODD copy in good state)

OK LePaRD artifact ready
  lepard_train_4000000_rev0194f95.jsonl  5.78 GB
  lepard_train_4000000_rev0194f95.jsonl.manifest.json  (449 B)
  lepard_train_4000000_rev0194f95.jsonl.sha256  (65 B)

⏱ Cell 7 - LePaRD ingestion completed in 1m 20.0s


In [10]:
# Cell 8: LePaRD ↔ CourtListener compatibility audit
"""
Purpose
-------
Pair-level compatibility audit between the LePaRD gold-standard citation
dataset and the CourtListener federal appellate corpus ingested in Cell 5.
Runs src.lepard_cl_compat as a module — pure analysis, no mutation of
either dataset. Deterministic and CI-gate capable (56 TDD tests).

What it measures
----------------
For every (citing_opinion, cited_opinion) pair in LePaRD, checks whether
BOTH endpoints exist in the CL corpus shards. Reports the usable gold
pair rate — the fraction of LePaRD pairs that can serve as ground truth
for retrieval / RAG evaluation on our CL corpus.

Why it matters
--------------
A low usable rate means the CL corpus is missing opinions LePaRD relies
on for its citation pairs, which would bias downstream retrieval eval.
This audit is the go/no-go check before Stage 4 (retrieval harness).

Output
------
Prints human-readable summary from src.lepard_cl_compat.__main__, including
match counts, usable pair rate, and any missing-endpoint diagnostics.
Raises SystemExit on non-zero exit code.

Runtime
-------
~1-3 seconds depending on CL corpus size and I/O throughput.
"""

import os, sys, subprocess, pathlib, time
os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

def _stream(cmd):
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, text=True, env=env,
    )
    for line in proc.stdout:
        sys.stdout.write(line); sys.stdout.flush()
    proc.wait()
    return proc.returncode

def _fmt_elapsed(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    m, s = divmod(seconds, 60)
    if m < 60:
        return f"{int(m)}m {s:.1f}s"
    h, m = divmod(m, 60)
    return f"{int(h)}h {int(m)}m {s:.1f}s"

_t_start = time.perf_counter()

print("=" * 60)
print("  src.lepard_cl_compat — pair-level compatibility audit")
print("=" * 60)
rc = _stream([".venv/bin/python", "-m", "src.lepard_cl_compat"])
if rc != 0:
    raise SystemExit(f"compat audit failed with exit code {rc}")

_elapsed = time.perf_counter() - _t_start
print("\nOK compatibility audit complete")
print(f"⏱ Cell 8 - LePaRD ↔ CL compatibility audit completed in {_fmt_elapsed(_elapsed)}")

  src.lepard_cl_compat — pair-level compatibility audit
LePaRD <-> CourtListener compatibility analysis

[1] ID-level overlap
  LePaRD unique ids:       512
  CL unique ids:           1,465,484
  Overlap:                 70 (13.7% of LePaRD)
  LePaRD id range max:     12,419,282
  CL id range max:         11,233,407
  LePaRD ids > CL max:     90 (heuristic: may indicate misaligned or differently-sourced id spaces)

[2] Pair-level overlap (both endpoints required for gold label)
  Total rows:              1,000
  Unique pairs:            454
  Unique sources / dests:  58 / 454
  Both endpoints in CL:    13 (2.9%)  <- USABLE GOLD
  Source only in CL:       105
  Dest only in CL:         40
  Neither in CL:           296

[3] Court distribution of matched CL ids
  Total matched with known court: 70
    ca9: 15
    ca5: 11
    ca4: 10
    ca11: 5
    ca8: 5
    ca3: 4
    cadc: 4
    cafc: 4
    ca1: 3
    ca10: 3
    ca6: 3
    ca2: 2
    ca7: 1

OK compatibility audit complete
⏱ Cell 8 -

In [11]:
# Cell 9: Data quality gate — NaN / encoding / parse audit on CL shards
"""
Purpose
-------
Post-probe data quality gate. Runs scripts/audit_jsonl_nan.py as a read-only
CI gate over all Drive-persistent shards produced by Cell 5 and (if needed)
repaired by Cell 5.5. Confirms steady-state cleanliness after Cell 6's
dataset readiness probe, providing an independent second measurement.

What it checks
--------------
- Bare NaN / Infinity literals that Polars' simd-json parser rejects
- Stringified NaN sentinels ("NaN", "nan", "Infinity", "-Infinity", "Inf", "-Inf")
- UTF-8 decode errors and malformed JSON lines
- Per-field contamination breakdown (case_name, raw_text, cleaning_flags, etc.)

Verdict semantics
-----------------
- CLEAN          : no contamination, shards ready for Stage 3
- REPAIRABLE     : NaN only in advisory fields (non-blocking)
- HARD_FAILURE   : NaN in required fields (blocks Stage 3)
- PARSE_FAILURE  : malformed JSON / decode errors — manual inspection required

This cell raises RuntimeError on any verdict other than CLEAN.

Runtime
-------
~30s for 1.46M rows on 48 CPU cores; ~90–110s on 12-core Colab A100 runtime.
"""

import os, sys, subprocess, json, pathlib, re, time
os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")
shard_dir = pathlib.Path("data/raw/cl_federal_appellate_bulk")
if not shard_dir.exists() or not any(shard_dir.glob("shard_*.jsonl")):
    raise RuntimeError(f"no shards under {shard_dir} — run Cell 5 first")

def _stream(cmd):
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, text=True, env=env,
    )
    lines = []
    for line in proc.stdout:
        sys.stdout.write(line); sys.stdout.flush()
        lines.append(line)
    proc.wait()
    return proc.returncode, "".join(lines)

def _extract_report(out: str) -> dict:
    # audit_jsonl_nan.py emits json.dumps(..., indent=2) — top-level braces
    # are anchored at column 0. Match line-anchored '{' ... '}' to avoid
    # false matches on nested empty objects like "nan_fields": {}.
    match = re.search(r'^\{.*?^\}', out, re.MULTILINE | re.DOTALL)
    if not match:
        raise RuntimeError("could not locate JSON report in audit output")
    return json.loads(match.group(0))

def _fmt_elapsed(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    m, s = divmod(seconds, 60)
    if m < 60:
        return f"{int(m)}m {s:.1f}s"
    h, m = divmod(m, 60)
    return f"{int(h)}h {int(m)}m {s:.1f}s"

_t_start = time.perf_counter()

print("=" * 60)
print("  scripts/audit_jsonl_nan.py --json (data quality gate)")
print("=" * 60)
rc, output = _stream([
    ".venv/bin/python", "scripts/audit_jsonl_nan.py",
    "--input-dir", str(shard_dir),
    "--json",
])
if rc != 0:
    raise SystemExit(f"audit failed with exit code {rc}")

report = _extract_report(output)
print("\n" + "=" * 60)
print("  Audit summary")
print("=" * 60)
print(f"  verdict:       {report.get('gate_verdict')}")
print(f"  clean_pct:     {report.get('clean_pct')}")
print(f"  total_lines:   {report.get('total_lines'):,}")
print(f"  total_shards:  {report.get('total_shards')}")
print(f"  nan_lines:     {report.get('nan_lines')}")
print(f"  nonfinite:     {report.get('nonfinite_lines')}")
print(f"  decode_errors: {report.get('decode_error_lines')}")
if report.get("gate_verdict") != "CLEAN":
    raise RuntimeError(f"audit gate failed: {report.get('gate_verdict')}")

_elapsed = time.perf_counter() - _t_start
print(f"\nOK all shards pass data quality gate")
print(f"⏱ Cell 9 - Data quality gate completed in {_fmt_elapsed(_elapsed)}")

  scripts/audit_jsonl_nan.py --json (data quality gate)
[INFO] Scanning 159 shards using 12 CPU cores ...

auditing: 100%|██████████| 159/159 [02:23<00:00,  1.11shard/s]
{
  "total_lines": 1456611,
  "nan_lines": 0,
  "nonfinite_lines": 0,
  "string_sentinel_lines": 0,
  "decode_error_lines": 0,
  "nan_shards": 0,
  "total_shards": 159,
  "clean_pct": 100.0,
  "nan_fields": {},
  "gate_verdict": "CLEAN",
  "contaminated_shards": []
}

  Audit summary
  verdict:       CLEAN
  clean_pct:     100.0
  total_lines:   1,456,611
  total_shards:  159
  nan_lines:     0
  nonfinite:     0
  decode_errors: 0

OK all shards pass data quality gate
⏱ Cell 9 - Data quality gate completed in 2m 23.5s


## Pipeline Summary — Stages 1–3 + Readiness Gates

End-to-end CourtListener + LePaRD data pipeline for the Legal RAG project. Every cell is thin orchestration over TDD-covered modules in `src/` and `scripts/` — no business logic lives in the notebook.

### Cell map

| Cell | Stage | Purpose | Modules / Scripts |
|---|---|---|---|
| **1** | — | Title and scope | (markdown) |
| **2** | Bootstrap | Clone repo, install `uv`, create `.venv` (Python 3.11.9), sync pinned deps, write `.env` | `uv`, `pyproject.toml`, `uv.lock` |
| **3** | Preflight | Reproducibility config + GPU/dependency contract + preflight gate | `src.repro`, `src.environment`, `src.timer` |
| **4** | Stage 1 (acq) | Discover and download 4 CourtListener bulk CSVs from public S3 (~62 GB), persist to Google Drive, skip if present | `src.config`, `src.s3_discovery`, `src.bulk_download` |
| **5** | Stages 1–2 | Filter chain (courts → dockets → clusters), extract 1.46M opinions to 159 JSONL shards, write manifest with SHA-256 checksums, run TDD contract tests | `src.pipeline` (`run_pipeline`, `validate_pipeline`), which chains `src.manifest`, `src.s3_discovery`, `src.bulk_download`, `src.filter_chain`, `src.extract`, `src.validation`, `src.schemas`, `src.exceptions` |
| **6** | Stage 3 readiness | Full-corpus Polars `scan_ndjson` + 8 gates (schema, A7, A8, A9, A11, A12, A13, B6) | `src.dataset_probe` (v2.5.11, 303 contract tests) |
| **7** | Stage 1 (LePaRD) | Ingest LePaRD 4M training pairs from Hugging Face at pinned revision `0194f95c…`, with sidecar + manifest; self-heals missing sidecars from disk bytes | `scripts/ingest_lepard.py` (79 TDD tests) |
| **8** | Readiness | LePaRD ↔ CourtListener compatibility audit — reports usable gold pair rate (both endpoints present in CL corpus) | `src.lepard_cl_compat` (56 TDD tests) |
| **9** | Data quality gate | NaN / encoding / parse audit over all shards; verdict: `CLEAN / REPAIRABLE / HARD_FAILURE / PARSE_FAILURE` | `scripts/audit_jsonl_nan.py` |

### Persistence strategy

Google Colab ephemeral storage is replaced with Google Drive symlinks so no stage repeats across sessions:

```
data/raw/cl_bulk                   -> /content/drive/MyDrive/cs1090b_cl_bulk
data/raw/cl_federal_appellate_bulk -> /content/drive/MyDrive/cs1090b_cl_federal_appellate_bulk
data/raw/lepard                    -> /content/drive/MyDrive/cs1090b_lepard
```

Fast-path behavior on reconnect:
- Cell 4: `download_file` skips any file already on Drive → ~0s
- Cell 5: `run_pipeline` reads manifest, validates shard SHA-256, returns immediately → ~1s
- Cell 6: Polars memory-mapped scan → ~30s regardless
- Cell 7: `--verify-only` checks sidecar + manifest + digest → ~5s
- Cell 9: 48-core parallel audit → ~30s

### Reproducibility guarantees

- **Python 3.11.9** pinned via `.python-version`
- **uv.lock** pins every dependency transitively (`torch 2.0.1+cu117`, `transformers 4.41.2`, `numpy 1.26.4`, …)
- **`src.repro.configure()`** sets `PYTHONHASHSEED=0`, `CUBLAS_WORKSPACE_CONFIG=:4096:8`, `torch.use_deterministic_algorithms(True)`, `cudnn.benchmark=False`, seeds Python/NumPy/torch CPU/CUDA RNGs to 0
- **LePaRD revision pinned** to `0194f95c3091acceab3b887c9b09ef432cf84052` (40-char SHA; mutable refs rejected)
- **Manifest checksums** — every shard's SHA-256 recorded; validation runs on every pipeline invocation
- **Contract tests** — `validate_pipeline` runs 13 `check_*` contract tests over sampled shards after extraction

### Downstream (not in this notebook)

README stages 4–7 — index generation (BM25 + FAISS), encoder training (BGE-M3 + Legal-BERT), sequential-loading evaluation (Tier A/B/C), W&B experiment tracking — are **not started** per the README and are correctly excluded from the data-pipeline notebook. They will be driven by `src.dataset_loader`, `src.lightning_datamodule`, `src.model_loader`, `src.split`, `src.wandb_logger`, and `src.hf_export` when that work begins.

### Verdict

> The full 1,465,484-record CourtListener Federal Appellate corpus plus the 4M-pair LePaRD training set are acquired, audited, probed, and verified compatible. The data pipeline is complete and reproducible across Colab sessions via Google Drive persistence. Ready for Stage 4 (indexing) whenever training begins.